In [9]:
# Explore dataset

import pandas as pd
data = pd.read_csv('D:\Real-Estate-AI-Assitant\data\cleaned_dataset.csv')

<>:4: SyntaxWarning: invalid escape sequence '\R'
<>:4: SyntaxWarning: invalid escape sequence '\R'
C:\Users\leona\AppData\Local\Temp\ipykernel_7228\3524964698.py:4: SyntaxWarning: invalid escape sequence '\R'
  data = pd.read_csv('D:\Real-Estate-AI-Assitant\data\cleaned_dataset.csv')


In [10]:
# Encode Categorical Variables

import sklearn.preprocessing as preprocessing
import sklearn.preprocessing as StandardScaler
label_encoder = preprocessing.LabelEncoder()
standard_scaler = StandardScaler.StandardScaler()

In [11]:
data['Lokasi']= label_encoder.fit_transform(data['Lokasi'])
data['Kota']= label_encoder.fit_transform(data['Kota'])


In [12]:
from sklearn.model_selection import train_test_split
Y = data['Harga']
X = data.drop('Harga', axis = 1)

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42, shuffle=True)

In [13]:
# Validation Set Creation
X_train, X_val, y_train, y_val = train_test_split(X_train,y_train, test_size=0.25, random_state=40, shuffle=True)  



,Lokasi,Kamar Tidur,Kamar Mandi,Garasi,Luas Tanah,Luas Bangunan,Kota
count,5994.000000,5994.00000,5994.000000,5994.000000,5.994000e+03,5.994000e+03,5994.000000
mean,61.761094,3.57007,2.838172,1.367534,-3.556270e-17,5.690032e-17,4.907407
std,38.630472,1.47220,1.162216,0.766750,1.000083e+00,1.000083e+00,2.573972
min,0.000000,1.00000,1.000000,0.000000,-7.471362e-01,-1.273449e+00,0.000000
25%,29.000000,3.00000,2.000000,1.000000,-3.833031e-01,-6.957321e-01,3.000000
50%,56.000000,3.00000,3.000000,1.000000,-1.888405e-01,-2.298317e-01,6.000000
75%,97.000000,4.00000,3.000000,2.000000,1.530372e-01,3.292487e-01,7.000000
max,124.000000,18.00000,7.000000,6.000000,1.696777e+01,4.708712e+00,8.000000


In [14]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
import numpy as np
import scipy.stats as stats

X_train[['Luas Tanah', 'Luas Bangunan']] = standard_scaler.fit_transform(X_train[['Luas Tanah', 'Luas Bangunan']]) # Only fit on training data to avoid data leakage

# Hyperparameter Tuning
rf = RandomForestRegressor(random_state=42)

# Define the hyperparameter grid
param_dist = {
    'n_estimators': stats.randint(50, 500),
    'max_depth': stats.randint(3, 20),
    'max_features' : ['sqrt', 'log2', None],
    'min_samples_split': stats.randint(2, 20),
    'min_samples_leaf': stats.randint(1, 20),
    'bootstrap': [True, False]
}



In [16]:
# Random Search CV
rf_random = RandomizedSearchCV(estimator = rf, param_distributions = param_dist, n_iter = 100, cv = 3, verbose=2, random_state=42, n_jobs = 1, scoring='neg_mean_squared_error', error_score='raise')
rf_random.fit(X_train, y_train)

print("Best Hyperparameters:", rf_random.best_params_)
tuned_model = rf_random.best_estimator_

Fitting 3 folds for each of 100 candidates, totalling 300 fits
[CV] END bootstrap=True, max_depth=17, max_features=None, min_samples_leaf=8, min_samples_split=8, n_estimators=171; total time=   0.3s
[CV] END bootstrap=True, max_depth=17, max_features=None, min_samples_leaf=8, min_samples_split=8, n_estimators=171; total time=   0.3s
[CV] END bootstrap=True, max_depth=17, max_features=None, min_samples_leaf=8, min_samples_split=8, n_estimators=171; total time=   0.3s
[CV] END bootstrap=True, max_depth=13, max_features=None, min_samples_leaf=4, min_samples_split=9, n_estimators=201; total time=   0.4s
[CV] END bootstrap=True, max_depth=13, max_features=None, min_samples_leaf=4, min_samples_split=9, n_estimators=201; total time=   0.4s
[CV] END bootstrap=True, max_depth=13, max_features=None, min_samples_leaf=4, min_samples_split=9, n_estimators=201; total time=   0.6s
[CV] END bootstrap=True, max_depth=4, max_features=log2, min_samples_leaf=6, min_samples_split=3, n_estimators=241; total

In [21]:
tuned_model.fit(X_train, y_train)
X_val[['Luas Tanah', 'Luas Bangunan']] = standard_scaler.transform(X_val[['Luas Tanah', 'Luas Bangunan']])

tuned_model.score(X_val, y_val)
print("Validation Set Score:", tuned_model.score(X_val, y_val))

#Test Data Evaluation
X_test[['Luas Tanah', 'Luas Bangunan']] = standard_scaler.transform(X_test[['Luas Tanah', 'Luas Bangunan']])
y_pred = tuned_model.predict(X_test)
from sklearn.metrics import mean_absolute_error, r2_score
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print("Test Set Mean Absolute Error:", mae)
print("Test Set R^2 Score:", r2)

Validation Set Score: -0.2396588778621409
Test Set Mean Absolute Error: 1619141570.8312316
Test Set R^2 Score: -0.2517928579882329


In [22]:
# Check the scale of your predictions vs actual values
print("y_test range:", y_test.min(), "to", y_test.max())
print("y_pred range:", y_pred.min(), "to", y_pred.max())
print("Mean of y_test:", y_test.mean())
print("Mean of y_pred:", y_pred.mean())

# Look for extreme errors
errors = y_test - y_pred
print("Max error:", errors.max())
print("Min error:", errors.min())

y_test range: 300000000 to 9900000000
y_pred range: 305000000.0 to 2810061551.155116
Mean of y_test: 2961407703.83992
Mean of y_pred: 1356009148.7933617
Max error: 8624034472.330017
Min error: -293765403.8297603


In [23]:
import joblib
import os
from datetime import datetime

# Create a timestamp for versioning
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Path 
path = 'D:/Real-Estate-AI-Assitant/models/'

# Save the model
model_filename = os.path.join(path, f'rf_model_{timestamp}.pkl')
joblib.dump(tuned_model, model_filename)


# Save model metadata
metadata_filename = os.path.join(path, f'metadata_{timestamp}.pkl')
metadata = {
    'model_type': 'RandomForestRegressor',
    'mae': mae,
    'r2_score': r2,
    'train_date': timestamp,
    'feature_names': list(X_train.columns),  # if using DataFrame
    'best_params': tuned_model.get_params()
}

joblib.dump(metadata, metadata_filename)

print(f"Model saved as: {model_filename}")

Model saved as: D:/Real-Estate-AI-Assitant/models/rf_model_20260107_162307.pkl
